In [ ]:
from pathlib import Path
import os

# notebooks live in ai/notebooks/; data paths are relative to ai/
os.chdir(Path("..").resolve())


# split the real images

All 7200 images are real. Kaggle already put them in Training and Testing for tumor classification, so I am not using that split.

Our split is 70% train, 15% val, 15% test, done separately inside each folder so glioma, meningioma, notumor and pituitary stay balanced. Seed is 42.

Fake copies come later, and each fake stays in the same split as its real image.

In [1]:
from pathlib import Path

import pandas as pd

data_dir = Path("data")
SEED = 42

rows = []
for split_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        for img_path in sorted(class_dir.glob("*.jpg")):
            rows.append({
                "path": img_path.as_posix(),
                "folder": class_dir.name,
                "kaggle_split": split_dir.name,
            })

images = pd.DataFrame(rows)
len(images)

7200

In [2]:
parts = []
for folder, group in images.groupby("folder"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(group)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    labels = ["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val)
    group = group.copy()
    group["split"] = labels
    parts.append(group)

split_df = pd.concat(parts, ignore_index=True)
split_df["seed"] = SEED
split_df.head()

,path,folder,kaggle_split,split,seed
0,data/Training/glioma/Tr-gl_810.jpg,glioma,Training,train,42
1,data/Training/glioma/Tr-gl_227.jpg,glioma,Training,train,42
2,data/Training/glioma/Tr-gl_160.jpg,glioma,Training,train,42
3,data/Testing/glioma/Te-gl_245.jpg,glioma,Testing,train,42
4,data/Training/glioma/Tr-gl_522.jpg,glioma,Training,train,42


In [3]:
split_df.groupby(["folder", "split"]).size().unstack(fill_value=0)

split,test,train,val
folder,,,
glioma,270,1260,270
meningioma,270,1260,270
notumor,270,1260,270
pituitary,270,1260,270


In [4]:
split_df["split"].value_counts()

split
train    5040
val      1080
test     1080
Name: count, dtype: int64

In [5]:
split_df.to_csv("metadata/split.csv", index=False)
print("saved", len(split_df), "rows")

saved 7200 rows
